# Dataset Splitting and Scaling

## Objective

Prepare the supervised AAPL dataset for model training.

The dataset is split chronologically into training, validation, and
test sets. No random shuffling is used because the observations
represent a temporal sequence.

Feature scaling is fitted exclusively on the training data and then
applied to validation and test data to prevent information leakage.

In [37]:
import numpy as np
import pandas as pd

X = np.load("../data/processed/X_sequences.npy")
y = np.load("../data/processed/y_labels.npy")

dates = pd.read_csv(
    "../data/processed/sample_dates.csv",
    index_col=0,
    parse_dates=True
).iloc[:, 0]

print("X:", X.shape)
print("y:", y.shape)
print("dates:", len(dates))

X: (200, 22, 9)
y: (200,)
dates: 200


In [38]:
n_samples = len(X)

train_end = int(n_samples * 0.60)
val_end = int(n_samples * 0.80)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

dates_train = dates.iloc[:train_end]
dates_val = dates.iloc[train_end:val_end]
dates_test = dates.iloc[val_end:]

In [39]:
print("TRAIN")
print(X_train.shape, y_train.shape)
print(dates_train.iloc[0], "→", dates_train.iloc[-1])

print("\nVALIDATION")
print(X_val.shape, y_val.shape)
print(dates_val.iloc[0], "→", dates_val.iloc[-1])

print("\nTEST")
print(X_test.shape, y_test.shape)
print(dates_test.iloc[0], "→", dates_test.iloc[-1])

TRAIN
(120, 22, 9) (120,)
2025-10-24 → 2026-04-17

VALIDATION
(40, 22, 9) (40,)
2026-04-20 → 2026-06-15

TEST
(40, 22, 9) (40,)
2026-06-16 → 2026-08-12


In [40]:
assert dates_train.iloc[-1] < dates_val.iloc[0]
assert dates_val.iloc[-1] < dates_test.iloc[0]

print("Chronological split verified.")

Chronological split verified.


## Feature Scaling

The nine input features have different numerical scales. Standardization
is therefore applied before model training.

To prevent information leakage, the scaler is fitted exclusively on
the training set. The same fitted transformation is then applied to
the validation and test sets.

In [41]:
from sklearn.preprocessing import StandardScaler

In [42]:
n_train, sequence_length, n_features = X_train.shape

scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_features)

scaler.fit(X_train_2d)

print("Scaler fitted on training data only.")

Scaler fitted on training data only.


In [43]:
X_train_scaled = scaler.transform(
    X_train.reshape(-1, n_features)
).reshape(X_train.shape)

X_val_scaled = scaler.transform(
    X_val.reshape(-1, n_features)
).reshape(X_val.shape)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(X_test.shape)

In [44]:
print("Original:")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nScaled:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

Original:
X_train: (120, 22, 9)
X_val: (40, 22, 9)
X_test: (40, 22, 9)

Scaled:
X_train_scaled: (120, 22, 9)
X_val_scaled: (40, 22, 9)
X_test_scaled: (40, 22, 9)


In [45]:
train_scaled_flat = X_train_scaled.reshape(-1, n_features)

print("Training means:")
print(train_scaled_flat.mean(axis=0))

print("\nTraining standard deviations:")
print(train_scaled_flat.std(axis=0))

Training means:
[-8.17853781e-16 -1.11905434e-15  6.91681561e-16 -3.87232334e-16
 -1.20875185e-13  4.50814470e-14 -2.19647532e-16  6.26485396e-14
 -2.22969791e-16]

Training standard deviations:
[1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [46]:
val_scaled_flat = X_val_scaled.reshape(-1, n_features)
test_scaled_flat = X_test_scaled.reshape(-1, n_features)

print("Validation means:")
print(val_scaled_flat.mean(axis=0))

print("\nTest means:")
print(test_scaled_flat.mean(axis=0))

Validation means:
[1.57855207 1.66028498 1.58639124 1.66894978 1.50970619 1.01725751
 1.08156516 1.03407692 1.09156904]

Test means:
[4.34896612 4.53270955 4.18455357 4.3908294  4.56177254 5.14324651
 0.94411053 3.36309987 0.42546857]


In [47]:
label_mapping = {
    "Fall": 0,
    "Rise": 1
}

y_train_encoded = np.array(
    [label_mapping[label] for label in y_train]
)

y_val_encoded = np.array(
    [label_mapping[label] for label in y_val]
)

y_test_encoded = np.array(
    [label_mapping[label] for label in y_test]
)

In [48]:
print("Training labels:", np.unique(y_train_encoded, return_counts=True))
print("Validation labels:", np.unique(y_val_encoded, return_counts=True))
print("Test labels:", np.unique(y_test_encoded, return_counts=True))

Training labels: (array([0, 1]), array([61, 59]))
Validation labels: (array([0, 1]), array([11, 29]))
Test labels: (array([0, 1]), array([18, 22]))


In [49]:
np.save(
    "../data/processed/X_train_scaled.npy",
    X_train_scaled
)

np.save(
    "../data/processed/X_val_scaled.npy",
    X_val_scaled
)

np.save(
    "../data/processed/X_test_scaled.npy",
    X_test_scaled
)

np.save(
    "../data/processed/y_train.npy",
    y_train_encoded
)

np.save(
    "../data/processed/y_val.npy",
    y_val_encoded
)

np.save(
    "../data/processed/y_test.npy",
    y_test_encoded
)

print("Scaled datasets saved successfully.")

Scaled datasets saved successfully.
